In [3]:
import pandas as pd
import numpy as np

In [1]:
from utils import load_prepare_decorte, load_prepare_karrierewege, load_prepare_decorte_esco

In [2]:
_, _, test_pairs = load_prepare_karrierewege(
                consider_all_subspans_of_len_at_least_2=True, 
                minus_last=False, 
                language='en', 
                max_rows=100)

len grouped 28


100%|██████████| 28/28 [00:00<00:00, 1140.38it/s]


len grouped 29


100%|██████████| 29/29 [00:00<00:00, 4179.17it/s]


len grouped 28


100%|██████████| 28/28 [00:00<00:00, 3409.81it/s]


In [3]:
import re

SEP_TOKEN = "<SEP>" 

def extract_raw_titles_from_doc(doc: str) -> List[str]:
        """Extract raw job titles from a document (handles both formatted and plain text).
        
        Note: Titles are normalized (lowercase + stripped) to match the job_skill_map format.
        """
        # For formatted documents: "esco role: cook\n description: ..."
        titles = re.findall(r"esco role: (.*?)\n", doc)
        if not titles:
            # Fallback: try regular role pattern (for history)
            titles = re.findall(r"role: (.*?)\n", doc)
        if not titles:
            # Fallback: assume plain title(s) with possible SEP_TOKEN
            titles = [t.strip() for t in doc.split(SEP_TOKEN) if t.strip()]
        # Normalize titles to match mapping file format (lowercase + stripped)
        return [t.strip().lower() for t in titles]

NameError: name 'List' is not defined

In [1]:
from typing import List
import re
SEP_TOKEN = "<SEP>"

def extract_job_titles_from_history(history_doc: str) -> List[str]:
    """Extract job titles from a history document.
    
    Handles both formatted documents (e.g., "role: cook\\n description: ...")
    and plain titles separated by SEP_TOKEN.
    
    Returns normalized (lowercase + stripped) titles.
    """
    # Try formatted "role: <title>\\n description: ..."
    titles = re.findall(r"role: (.*?)\n", history_doc)
    
    # Fallback to "esco role: <title>\\n" format
    if not titles:
        titles = re.findall(r"esco role: (.*?)\n", history_doc)
    
    # Fallback to plain title(s) with SEP_TOKEN
    if not titles:
        titles = [t.strip() for t in history_doc.split(SEP_TOKEN) if t.strip()]
    
    # Normalize: lowercase + stripped
    return [t.strip().lower() for t in titles if t.strip()]

In [44]:
test_pairs

[("role: domestic cleaner \n description: Domestic cleaners perform all necessary cleaning activities in order to clean their clients' houses. They vacuum and sweep floors, wash dishes, launder clothes, dust, scrub and polish surfaces and disinfect equipment and materials.<SEP>role: domestic cleaner \n description: Domestic cleaners perform all necessary cleaning activities in order to clean their clients' houses. They vacuum and sweep floors, wash dishes, launder clothes, dust, scrub and polish surfaces and disinfect equipment and materials.",
  "esco role: domestic cleaner \n description: Domestic cleaners perform all necessary cleaning activities in order to clean their clients' houses. They vacuum and sweep floors, wash dishes, launder clothes, dust, scrub and polish surfaces and disinfect equipment and materials."),
 ("role: domestic cleaner \n description: Domestic cleaners perform all necessary cleaning activities in order to clean their clients' houses. They vacuum and sweep fl

In [46]:
from tqdm import tqdm

for i, (history_doc, target_doc) in enumerate(tqdm(test_pairs, desc="  Scoring")):
    job_titles = extract_job_titles_from_history(history_doc)
    print(job_titles)
    if i > 5:
        break

  Scoring:   3%|▎         | 6/192 [00:00<00:00, 11295.25it/s]

['domestic cleaner', 'domestic cleaner']
['domestic cleaner', 'domestic cleaner']
['domestic cleaner', 'domestic cleaner']
['domestic cleaner', 'street food vendor']
['street food vendor', 'wood caulker']
['wood caulker', 'garden labourer']
['garden labourer', 'wood caulker']


In [1]:
import pickle
import pandas as pd
import numpy as np

# Load the skill overlap scores
with open(r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/decorte/skill_overlap_scores_top100_isco_fused/test_scores_skill_overlap.pkl', 'rb') as f:
    skill_overlap_data = pickle.load(f)

# Display the structure and contents
print("Keys in the loaded data:")
print(skill_overlap_data.keys())
print("\n" + "="*80 + "\n")

# Display information about each key
for key, value in skill_overlap_data.items():
    print(f"Key: {key}")
    print(f"Type: {type(value)}")
    
    if isinstance(value, np.ndarray):
        print(f"Shape: {value.shape}")
        print(f"Dtype: {value.dtype}")
        print(f"Sample values (first 5): {value.flat[:5]}")
    elif isinstance(value, list):
        print(f"Length: {len(value)}")
        print(f"Sample values (first 3): {value[:3]}")
    else:
        print(f"Value: {value}")
    
    print("-" * 80)

# If scores exist, show basic statistics
if 'scores' in skill_overlap_data:
    scores = skill_overlap_data['scores']
    print("\nSkill Overlap Scores Statistics:")
    print(f"Min score: {np.min(scores):.4f}")
    print(f"Max score: {np.max(scores):.4f}")
    print(f"Mean score: {np.mean(scores):.4f}")
    print(f"Median score: {np.median(scores):.4f}")
    print(f"Std dev: {np.std(scores):.4f}")


Keys in the loaded data:
dict_keys(['scores', 'target_labels', 'true_target_indices', 'histories', 'true_targets', 'split'])


Key: scores
Type: <class 'numpy.ndarray'>
Shape: (1802, 1022)
Dtype: float32
Sample values (first 5): [0.         0.         0.         0.33333334 0.01754386]
--------------------------------------------------------------------------------
Key: target_labels
Type: <class 'list'>
Length: 1022
Sample values (first 3): ['3d modeller ', 'academic advisor ', 'academic support officer ']
--------------------------------------------------------------------------------
Key: true_target_indices
Type: <class 'list'>
Length: 1802
Sample values (first 3): [824, 431, 431]
--------------------------------------------------------------------------------
Key: histories
Type: <class 'list'>
Length: 1802
Sample values (first 3): ['Chef de Cuisine ', 'Chef/General Manager ', 'Chef de Cuisine <SEP>Chef/General Manager ']
------------------------------------------------------------

In [2]:
# new scores

import pickle
import pandas as pd
import numpy as np

# Load the skill overlap scores
with open(r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/decorte/skill_overlap_scores_top100_isco_fused_2/test_scores_skill_overlap.pkl', 'rb') as f:
    skill_overlap_data = pickle.load(f)

# Display the structure and contents
print("Keys in the loaded data:")
print(skill_overlap_data.keys())
print("\n" + "="*80 + "\n")

# Display information about each key
for key, value in skill_overlap_data.items():
    print(f"Key: {key}")
    print(f"Type: {type(value)}")
    
    if isinstance(value, np.ndarray):
        print(f"Shape: {value.shape}")
        print(f"Dtype: {value.dtype}")
        print(f"Sample values (first 5): {value.flat[:5]}")
    elif isinstance(value, list):
        print(f"Length: {len(value)}")
        print(f"Sample values (first 3): {value[:3]}")
    else:
        print(f"Value: {value}")
    
    print("-" * 80)

# If scores exist, show basic statistics
if 'scores' in skill_overlap_data:
    scores = skill_overlap_data['scores']
    print("\nSkill Overlap Scores Statistics:")
    print(f"Min score: {np.min(scores):.4f}")
    print(f"Max score: {np.max(scores):.4f}")
    print(f"Mean score: {np.mean(scores):.4f}")
    print(f"Median score: {np.median(scores):.4f}")
    print(f"Std dev: {np.std(scores):.4f}")


Keys in the loaded data:
dict_keys(['scores', 'target_labels', 'true_target_indices', 'histories', 'true_targets', 'job_ids', 'split'])


Key: scores
Type: <class 'numpy.ndarray'>
Shape: (1802, 1022)
Dtype: float32
Sample values (first 5): [0.         0.         0.         0.30555555 0.03508772]
--------------------------------------------------------------------------------
Key: target_labels
Type: <class 'list'>
Length: 1022
Sample values (first 3): ['3d modeller ', 'academic advisor ', 'academic support officer ']
--------------------------------------------------------------------------------
Key: true_target_indices
Type: <class 'list'>
Length: 1802
Sample values (first 3): [824, 431, 431]
--------------------------------------------------------------------------------
Key: histories
Type: <class 'list'>
Length: 1802
Sample values (first 3): ['Chef de Cuisine ', 'Chef/General Manager ', 'Chef de Cuisine <SEP>Chef/General Manager ']
-------------------------------------------------

In [3]:
def calculate_ranking_metrics(scores, true_indices, k_values=[1, 5, 10, 20]):
    """
    Calculate MRR and Recall@K metrics.
    """
    n_samples = len(true_indices)
    rr_sum = 0
    hits_at_k = {k: 0 for k in k_values}
    valid_samples = 0
    
    for i, true_idx in enumerate(true_indices):
        if true_idx < 0: # Skip unmapped targets
            continue
            
        valid_samples += 1
        sample_scores = scores[i]
        
        # Sort indices in descending order of score
        sorted_indices = np.argsort(-sample_scores)
        
        # Calculate Rank (1-indexed)
        rank = np.where(sorted_indices == true_idx)[0][0] + 1
        rr_sum += 1.0 / rank
        
        # Calculate Hits@K
        for k in k_values:
            if true_idx in sorted_indices[:k]:
                hits_at_k[k] += 1
                
    metrics = {
        'MRR': rr_sum / valid_samples if valid_samples > 0 else 0.0
    }
    for k in k_values:
        metrics[f'R@{k}'] = hits_at_k[k] / valid_samples if valid_samples > 0 else 0.0
        
    return metrics

# Calculate metrics if the required keys exist
if 'scores' in skill_overlap_data and 'true_target_indices' in skill_overlap_data:
    scores = skill_overlap_data['scores']
    true_indices = skill_overlap_data['true_target_indices']
    
    metrics = calculate_ranking_metrics(scores, true_indices)
    
    print("\nRanking Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("Required keys ('scores' and 'true_target_indices') not found in data.")
    print(f"Available keys: {list(skill_overlap_data.keys())}")


Ranking Metrics:
MRR: 0.1621
R@1: 0.0694
R@5: 0.2647
R@10: 0.3518
R@20: 0.4262


In [11]:
with open(r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/decorte/skill_overlap_scores_top100_isco_fused_2/test_scores_skill_overlap.pkl', 'rb') as f:
    skill_overlap_data = pickle.load(f)

# Calculate metrics if the required keys exist
if 'scores' in skill_overlap_data and 'true_target_indices' in skill_overlap_data:
    scores = skill_overlap_data['scores']
    true_indices = skill_overlap_data['true_target_indices']
    
    metrics = calculate_ranking_metrics(scores, true_indices)
    
    print("\nRanking Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("Required keys ('scores' and 'true_target_indices') not found in data.")
    print(f"Available keys: {list(skill_overlap_data.keys())}")


Ranking Metrics:
MRR: 0.1621
R@1: 0.0694
R@5: 0.2647
R@10: 0.3518
R@20: 0.4262


In [10]:
with open(r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/decorte/skill_overlap_scores_top100_isco_fused/test_scores_skill_overlap.pkl', 'rb') as f:
    skill_overlap_data = pickle.load(f)

# Calculate metrics if the required keys exist
if 'scores' in skill_overlap_data and 'true_target_indices' in skill_overlap_data:
    scores = skill_overlap_data['scores']
    true_indices = skill_overlap_data['true_target_indices']
    
    metrics = calculate_ranking_metrics(scores, true_indices)
    
    print("\nRanking Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("Required keys ('scores' and 'true_target_indices') not found in data.")
    print(f"Available keys: {list(skill_overlap_data.keys())}")


Ranking Metrics:
MRR: 0.1738
R@1: 0.0794
R@5: 0.2858
R@10: 0.3563
R@20: 0.4312


In [ ]:
with open(r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/decorte/skill_overlap_scores_top100_isco_fused/test_scores_skill_overlap.pkl', 'rb') as f:
    skill_overlap_data = pickle.load(f)

# Calculate metrics if the required keys exist
if 'scores' in skill_overlap_data and 'true_target_indices' in skill_overlap_data:
    scores = skill_overlap_data['scores']
    true_indices = skill_overlap_data['true_target_indices']
    
    metrics = calculate_ranking_metrics(scores, true_indices)
    
    print("\nRanking Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
else:
    print("Required keys ('scores' and 'true_target_indices') not found in data.")
    print(f"Available keys: {list(skill_overlap_data.keys())}")

In [ ]:
# Load the text scores
with open('/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/kw_esco_100k_ablation/job_titles_desc/scores/test_scores_text.pkl', 'rb') as f:
    text_scores_data = pickle.load(f)

# Display the structure and contents
print("Keys in the loaded data:")
print(text_scores_data.keys())
print("\n" + "="*80 + "\n")

# Display information about each key
for key, value in text_scores_data.items():
    print(f"Key: {key}")
    print(f"Type: {type(value)}")
    
    if isinstance(value, np.ndarray):
        print(f"Shape: {value.shape}")
        print(f"Dtype: {value.dtype}")
        print(f"Sample values (first 5): {value.flat[:5]}")
    elif isinstance(value, list):
        print(f"Length: {len(value)}")
        print(f"Sample values (first 3): {value[:3]}")
    else:
        print(f"Value: {value}")
    
    print("-" * 80)

# If scores exist, show basic statistics
if 'scores' in text_scores_data:
    scores = text_scores_data['scores']
    print("\nText Scores Statistics:")
    print(f"Min score: {np.min(scores):.4f}")
    print(f"Max score: {np.max(scores):.4f}")
    print(f"Mean score: {np.mean(scores):.4f}")
    print(f"Median score: {np.median(scores):.4f}")
    print(f"Std dev: {np.std(scores):.4f}")


Keys in the loaded data:
dict_keys(['scores', 'target_labels', 'true_target_indices', 'split', 'histories', 'true_targets'])


Key: scores
Type: <class 'numpy.ndarray'>
Shape: (137530, 1186)
Dtype: float32
Sample values (first 5): [0.7187167  0.5633316  0.49566773 0.50317025 0.47062084]
--------------------------------------------------------------------------------
Key: target_labels
Type: <class 'list'>
Length: 1186
Sample values (first 3): ['esco role: materials engineer \n description: Materials engineers research and design new or improved materials for a diverse number of applications. They analyse the composition of materials, conduct experiments, and develop new materials for industry-specific use that can range from rubber, to textiles, glass, metals, and chemicals. They advise companies in damage assessments, quality assurance of materials, and recycling of materials.', "esco role: data centre operator \n description: Data centre operators maintain computer operations within 

In [5]:
from typing import Dict, List, Tuple, Optional
import os
import pickle
import sys

def load_scores(score_path: str) -> Dict:
    """Load score dictionary from pickle file."""
    with open(score_path, 'rb') as f:
        return pickle.load(f)

In [6]:
text_test = load_scores(os.path.join(
    "/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/cpp/kw_esco_100k_ablation/job_titles_desc/scores", 
    "test_scores_text.pkl"
    ))

skill_test = load_scores(os.path.join(
    "../src/cpp/decorte/results", 
    "test_scores_skill_overlap.pkl"))



In [7]:
# Calculate ranking metrics for skill_test
def calculate_ranking_metrics(scores: np.ndarray, true_target_indices: List[int],
                              k_values: List[int] = [1, 5, 10, 20]) -> Dict[str, float]:
    """
    Calculate ranking metrics from score matrix.
    
    Args:
        scores: Score matrix [n_samples, n_targets]
        true_target_indices: Index of true target for each sample
        k_values: List of K values for Recall@K
        
    Returns:
        Dictionary with MRR and Recall@K metrics
    """
    n_samples = len(true_target_indices)
    
    if n_samples == 0:
        return {'MRR': 0.0, **{f'R@{k}': 0.0 for k in k_values}}
    
    reciprocal_ranks = []
    hits_at_k = {k: 0 for k in k_values}
    
    # Sort indices in descending order of score
    sorted_indices = np.argsort(scores, axis=1)[:, ::-1]
    
    for i, true_idx in enumerate(true_target_indices):
        if true_idx < 0:
            continue
        
        # Find rank of true target (1-indexed)
        rank = np.where(sorted_indices[i] == true_idx)[0]
        if len(rank) > 0:
            rank = rank[0] + 1
            reciprocal_ranks.append(1.0 / rank)
            
            # Check hits at k
            for k in k_values:
                if rank <= k:
                    hits_at_k[k] += 1
    
    # Calculate metrics
    mrr = np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0
    recall_at_k = {k: hits_at_k[k] / n_samples for k in k_values}
    
    return {
        'MRR': mrr,
        **{f'R@{k}': recall_at_k[k] for k in k_values}
    }

# Calculate metrics for skill_test
skill_metrics = calculate_ranking_metrics(
    skill_test['scores'],
    skill_test['true_target_indices'],
    k_values=[5, 10, 20]
)

print("Skill Overlap Test Set Metrics:")
print("=" * 80)
print(f"MRR:   {skill_metrics['MRR']:.4f}")
print(f"R@5:   {skill_metrics['R@5']:.4f}")
print(f"R@10:  {skill_metrics['R@10']:.4f}")
print(f"R@20:  {skill_metrics['R@20']:.4f}")


: 

In [28]:
text_test.keys()

dict_keys(['scores', 'target_labels', 'true_target_indices', 'split', 'histories', 'true_targets'])

In [48]:
# Print everything for the first sample
print("First sample analysis:")
print("=" * 80)

# History
print("\n1. History:")
print(text_test['histories'][0])

# True target
print("\n2. True target:")
print(text_test['true_targets'][0])

# True target index
print("\n3. True target index:")
print(text_test['true_target_indices'][0])

# Scores for all targets
print("\n4. Scores for all targets (first sample):")
print(text_test['scores'][0])

# Top 10 predictions
print("\n5. Top 10 predictions:")
scores_first = text_test['scores'][0]
top_10_indices = np.argsort(scores_first)[::-1][:10]
for rank, idx in enumerate(top_10_indices, 1):
    print(f"  Rank {rank}: {text_test['target_labels'][idx]} (score: {scores_first[idx]:.4f})")

# Check if true target is in top 10
true_idx = text_test['true_target_indices'][0]
if true_idx >= 0:
    true_rank = np.where(np.argsort(scores_first)[::-1] == true_idx)[0][0] + 1
    print(f"\n6. True target rank: {true_rank}")
    print(f"   True target score: {scores_first[true_idx]:.4f}")
else:
    print("\n6. True target not found in target labels")

First sample analysis:

1. History:
role: sales engineer 
 description: Sales engineers provide technical customisation of products based on customers requests and needs (mainly heavy duty), such as building equipment. They take care of business to business contact and assume responsibility for complex repairs and maintenance process.

2. True target:
esco role: sales engineer 
 description: Sales engineers provide technical customisation of products based on customers requests and needs (mainly heavy duty), such as building equipment. They take care of business to business contact and assume responsibility for complex repairs and maintenance process.

3. True target index:
222

4. Scores for all targets (first sample):
[0.7187167  0.5633316  0.49566773 ... 0.46738383 0.34392986 0.5686762 ]

5. Top 10 predictions:
  Rank 1: esco role: blacksmith 
 description: Blacksmiths heat metal, usually steel, in a forge and shape it with a hammer, chisel, and an anvil. Contemporarily, they predom

In [49]:
scores_first

array([0.7187167 , 0.5633316 , 0.49566773, ..., 0.46738383, 0.34392986,
       0.5686762 ], shape=(1186,), dtype=float32)

In [54]:
text_test['target_labels'][223]

'esco role: sales engineer \n description: Sales engineers provide technical customisation of products based on customers requests and needs (mainly heavy duty), such as building equipment. They take care of business to business contact and assume responsibility for complex repairs and maintenance process.'

In [51]:
true_idx

222

In [13]:
text_scores = text_test.copy()
skill_scores = skill_test.copy()

In [6]:
text_targets = text_test['target_labels']
skill_targets = skill_test['target_labels']

In [29]:
print(f"text_targets length: {len(text_test['histories'])}")
print(f"skill_targets length: {len(skill_test['histories'])}")

text_targets length: 137530
skill_targets length: 137530


In [30]:
skill_test['histories'][:10]

['sales engineer ',
 'sales engineer ',
 'sales engineer <SEP>sales engineer ',
 'building construction worker ',
 'mover ',
 'inventory coordinator ',
 'building construction worker <SEP>mover ',
 'mover <SEP>inventory coordinator ',
 'building construction worker <SEP>mover <SEP>inventory coordinator ',
 'foundry operative ']

In [31]:
text_test['histories'][:10]

['role: sales engineer \n description: Sales engineers provide technical customisation of products based on customers requests and needs (mainly heavy duty), such as building equipment. They take care of business to business contact and assume responsibility for complex repairs and maintenance process.',
 'role: sales engineer \n description: Sales engineers provide technical customisation of products based on customers requests and needs (mainly heavy duty), such as building equipment. They take care of business to business contact and assume responsibility for complex repairs and maintenance process.',
 'role: sales engineer \n description: Sales engineers provide technical customisation of products based on customers requests and needs (mainly heavy duty), such as building equipment. They take care of business to business contact and assume responsibility for complex repairs and maintenance process.<SEP>role: sales engineer \n description: Sales engineers provide technical customisa

In [9]:
skill_target_to_idx = {t: i for i, t in enumerate(skill_targets)}

In [10]:
skill_target_to_idx

{'video and motion picture producer ': 0,
 'scenic painter ': 1,
 'ict research manager ': 2,
 'precision engineer ': 3,
 'ornamental metal worker ': 4,
 'call centre agent ': 5,
 'camping ground manager ': 6,
 'assistant stage director ': 7,
 'wind musical instrument maker ': 8,
 'philosopher ': 9,
 'laundry worker ': 10,
 'building materials specialised seller ': 11,
 'industrial machinery mechanic ': 12,
 'blockchain developer ': 13,
 'assistant lecturer ': 14,
 'bee breeder ': 15,
 'amusement and recreation attendant ': 16,
 'employment agent ': 17,
 'geothermal power plant operator ': 18,
 'rail construction supervisor ': 19,
 'religion scientific researcher ': 20,
 'assistant video and motion picture director ': 21,
 'securities analyst ': 22,
 'file clerk ': 23,
 'aircraft pilot ': 24,
 'process engineer ': 25,
 'information manager ': 26,
 'architectural drafter ': 27,
 'medical laboratory technology vocational teacher ': 28,
 'precision mechanics supervisor ': 29,
 'foundry op

In [14]:
# Reorder skill scores columns
n_samples = skill_scores['scores'].shape[0]
n_text_targets = len(text_targets)
aligned_skill_scores = np.zeros((n_samples, n_text_targets), dtype=np.float32)

In [16]:
aligned_skill_scores

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(137530, 1186), dtype=float32)

In [36]:
for i, target in enumerate(text_targets):
        match_idx = -1
        
        # 1. Try direct match
        if target in skill_target_to_idx:
            match_idx = skill_target_to_idx[target]
        
        # 2. Try extracting title from "esco role: <title>\n..." format
        else:
            clean_target = target
            if "role: " in target:
                # Handle "esco role: ..." or just "role: ..."
                # Split by "role: ", take the part after, then split by newline to get title
                try:
                    clean_target = target.split("role: ", 1)[1].split("\n", 1)[0].strip()
                    clean_target = clean_target + " "
                    if i < 5:
                        print(clean_target)
                except IndexError:
                    pass
            
            if clean_target in skill_target_to_idx:
                match_idx = skill_target_to_idx[clean_target]
        
        if match_idx != -1:
            aligned_skill_scores[:, i] = skill_scores['scores'][:, match_idx]
        else:
            # Log first few characters to debug (limit to first 5 unmatched to avoid spam)
            if i < 5: 
                print(f"    Target '{target[:30]}...' not found in skill scores (tried clean: '{clean_target if 'clean_target' in locals() else target}')")

materials engineer 
data centre operator 
ict account manager 
aircraft marshaller 
general veterinarian 


In [37]:
aligned_skill_scores

array([[0.0212766 , 0.        , 0.15789473, ..., 0.        , 0.        ,
        0.        ],
       [0.0212766 , 0.        , 0.15789473, ..., 0.        , 0.        ,
        0.        ],
       [0.0212766 , 0.        , 0.15789473, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.06382979, 0.        , 0.        , ..., 0.02439024, 0.44      ,
        0.        ],
       [0.06382979, 0.        , 0.        , ..., 0.02439024, 0.36      ,
        0.        ],
       [0.06382979, 0.        , 0.        , ..., 0.02439024, 0.44      ,
        0.        ]], shape=(137530, 1186), dtype=float32)